# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mhassanbuilds/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [21]:
# ML-09 setup
import sys
import os
import numpy as np
import pandas as pd

print("Python and core libraries loaded.")

Python and core libraries loaded.


In [22]:
# Load the official FlyRank repository and prepare the Week-5 data

import subprocess
from pathlib import Path

repo = Path("/content/flyrank-reference")

# Clone the official FlyRank repository
if not repo.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "-q",
            "https://github.com/flyrank-bih/flyrank-ml-internship-starter.git",
            str(repo)
        ],
        check=True
    )

# Make the official FlyRank scripts importable
scripts_path = str(repo / "scripts")

if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

print("Official FlyRank repository ready.")
print("Scripts path:", scripts_path)

# Prepare official feature vector
subprocess.run(
    ["python", "scripts/01_prepare_features.py"],
    cwd=repo,
    check=True
)

# Create official Week-4 baseline scores
subprocess.run(
    ["python", "scripts/02_baseline_score.py"],
    cwd=repo,
    check=True
)

# Define processed data paths
feature_path = (
    repo / "data" / "processed" / "refresh_feature_vector.csv"
)

baseline_path = (
    repo / "data" / "processed" / "baseline_refresh_queue.csv"
)

# Load processed data
features_df = pd.read_csv(feature_path)
baseline_df = pd.read_csv(baseline_path)

print("Feature vector shape:", features_df.shape)
print("Baseline shape:", baseline_df.shape)

Official FlyRank repository ready.
Scripts path: /content/flyrank-reference/scripts
Feature vector shape: (30000, 52)
Baseline shape: (30000, 22)


In [23]:
# Load the official FlyRank modeling utilities

import importlib.util

script_path = repo / "scripts" / "03_train_model.py"

spec = importlib.util.spec_from_file_location(
    "flyrank_train_model",
    script_path
)

flyrank_model = importlib.util.module_from_spec(spec)

sys.modules["flyrank_train_model"] = flyrank_model

spec.loader.exec_module(flyrank_model)

print("Official FlyRank modeling utilities loaded.")

Official FlyRank modeling utilities loaded.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Growing pages were younger

The FlyRank research reports that growing pages were younger on average than declining pages. The reported averages were approximately 185 days for growing pages and 228 days for declining pages, while word count was nearly the same. This is useful as an observed association, but it does not by itself show that page age causes growth or decline.

**My methodology question:** Where exactly does the growing/declining label come from, and how is the time window for that label defined? I would want to confirm that the label is constructed independently from the predictor measurements so that information from the outcome period is not unintentionally included in the analysis.

### Finding 2 — Refreshed pages showed positive lift

The FlyRank research reports positive refresh lift across several groups and reports a positive median effect for refreshed versus stale pages among older pages. This is an observed difference between the groups rather than automatic proof that refreshing caused the improvement.

**My methodology question:** How were pages selected for refreshing, and how does the validation design account for pre-existing differences between refreshed and non-refreshed pages? Pages selected for refresh may already differ in quality, traffic, age, or opportunity, so I would want to understand how the analysis separates the observed refresh association from those possible differences.

Overall, these questions are intended as constructive checks on how the labels and validation design support the reported findings. They do not mean that the findings are incorrect; they identify what I would want to understand before making a stronger claim.


In [24]:
# Section 1 check
paper_findings = [
    "Growing pages were younger on average than declining pages.",
    "Refreshed pages showed positive observed lift in the reported analysis."
]

methodology_questions = [
    "How is the growing/declining label constructed and separated from predictor measurement?",
    "How are pre-existing differences between refreshed and non-refreshed pages handled?"
]

print("Paper findings documented:", len(paper_findings))
print("Methodology questions documented:", len(methodology_questions))

assert len(paper_findings) == 2
assert len(methodology_questions) == 2

print("Section 1 check passed.")

Paper findings documented: 2
Methodology questions documented: 2
Section 1 check passed.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


I re-ran the Week-5 Random Forest under two validation designs. The before condition uses a row-level random split, where rows from the same client can appear in both training and test sets. The after condition uses the official client-aware holdout, so clients in the test set are not used for training.

The row-level random split measured Precision@20 of **0.95**, Precision@50 of **0.90**, and Precision@100 of **0.90**. Under the client-aware split, the measured values were **0.65**, **0.74**, and **0.72**, respectively.

The lower measurements under the client-aware split provide a more conservative assessment of how the model's ranking signal transfers to held-out clients. The difference does not by itself prove leakage or explain the entire performance change, so I investigate the feature set separately in the leakage audit.

These results should be interpreted as measured performance under the tested validation designs, not as a guarantee of performance on future clients or datasets.


In [25]:
# Section 2 — My model under an honest split (before/after)

from sklearn.model_selection import train_test_split

# Load the official processed data
frame = pd.read_csv(feature_path)
baseline_frame = pd.read_csv(baseline_path)

print("Rows:", len(frame))
print("Features:", len(frame.columns))

# Target
target_series = frame["is_declining_label"].astype(int)

print(
    "Target positive rows:",
    int(target_series.sum())
)

print(
    "Target positive rate:",
    round(target_series.mean(), 4)
)

# Build the official feature matrix
feature_frame, feature_columns = (
    flyrank_model.build_feature_matrix(frame)
)

print("Model feature matrix shape:", feature_frame.shape)


# =========================================================
# BEFORE: row-level random split
# =========================================================

all_indices = np.arange(len(frame))

before_train_idx, before_test_idx = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=42,
    stratify=target_series
)

print("\nBEFORE — Row-level random split")
print("Training rows:", len(before_train_idx))
print("Test rows:", len(before_test_idx))

before_train_features = feature_frame.iloc[before_train_idx]
before_test_features = feature_frame.iloc[before_test_idx]

before_train_target = target_series.iloc[before_train_idx]
before_test_target = target_series.iloc[before_test_idx]

# Train Random Forest
before_model = flyrank_model.build_models()["random_forest"]

before_model.fit(
    before_train_features,
    before_train_target
)

before_probabilities = flyrank_model.predict_probability(
    before_model,
    before_test_features
)

before_metrics = flyrank_model.metric_payload(
    before_test_target,
    before_probabilities
)


# =========================================================
# AFTER: official client-aware split
# =========================================================

after_train_idx, after_test_idx, split_strategy = (
    flyrank_model.make_client_aware_split(
        frame,
        target_series
    )
)

print("\nAFTER — Client-aware split")
print("Split strategy:", split_strategy)
print("Training rows:", len(after_train_idx))
print("Test rows:", len(after_test_idx))

after_train_features = feature_frame.iloc[after_train_idx]
after_test_features = feature_frame.iloc[after_test_idx]

after_train_target = target_series.iloc[after_train_idx]
after_test_target = target_series.iloc[after_test_idx]

# Train Random Forest
after_model = flyrank_model.build_models()["random_forest"]

after_model.fit(
    after_train_features,
    after_train_target
)

after_probabilities = flyrank_model.predict_probability(
    after_model,
    after_test_features
)

after_metrics = flyrank_model.metric_payload(
    after_test_target,
    after_probabilities
)


# =========================================================
# BEFORE vs AFTER
# =========================================================

validation_comparison = pd.DataFrame([
    {
        "Validation": "Before: row-level random split",
        "Precision@20": before_metrics["precision_at_20"],
        "Precision@50": before_metrics["precision_at_50"],
        "Precision@100": before_metrics["precision_at_100"],
    },
    {
        "Validation": "After: client-aware split",
        "Precision@20": after_metrics["precision_at_20"],
        "Precision@50": after_metrics["precision_at_50"],
        "Precision@100": after_metrics["precision_at_100"],
    }
])

for column in [
    "Precision@20",
    "Precision@50",
    "Precision@100"
]:
    validation_comparison[column] = (
        validation_comparison[column].round(3)
    )

print("\nBefore vs After:")
display(validation_comparison)

Rows: 30000
Features: 52
Target positive rows: 16262
Target positive rate: 0.5421
Model feature matrix shape: (30000, 52)

BEFORE — Row-level random split
Training rows: 24000
Test rows: 6000

AFTER — Client-aware split
Split strategy: client_holdout
Training rows: 27675
Test rows: 2325

Before vs After:


,Validation,Precision@20,Precision@50,Precision@100
0,Before: row-level random split,0.95,0.90,0.90
1,After: client-aware split,0.65,0.74,0.72


### Model failure examples

To inspect where the model can make mistakes, I reviewed examples from the client-aware test set where the Random Forest's predicted class disagreed with the observed label.

I focus on high-confidence disagreements because they are useful for understanding where the model's ranking signal can fail. These examples are not used to claim a general error pattern; they are concrete observations from the held-out test set.

The examples show that a high model score does not guarantee that a page belongs to the target class. This supports using the model as decision-support for prioritization rather than as an automatic decision rule.

In [26]:
# Section 2 — Real failure examples

# Build a table of client-aware test predictions
failure_examples = frame.iloc[after_test_idx].copy()

failure_examples["actual_label"] = after_test_target.to_numpy()
failure_examples["predicted_probability"] = after_probabilities
failure_examples["predicted_label"] = (
    failure_examples["predicted_probability"] >= 0.5
).astype(int)

# Keep only incorrect predictions
failure_examples = failure_examples[
    failure_examples["actual_label"]
    != failure_examples["predicted_label"]
].copy()

# Confidence = distance from 0.5
failure_examples["confidence"] = (
    failure_examples["predicted_probability"] - 0.5
).abs()

# Show the highest-confidence mistakes
failure_examples = failure_examples.sort_values(
    "confidence",
    ascending=False
)

print("Total incorrect predictions:", len(failure_examples))

display(
    failure_examples[
        [
            "actual_label",
            "predicted_label",
            "predicted_probability",
            "confidence"
        ]
    ].head(10)
)

Total incorrect predictions: 762


,actual_label,predicted_label,predicted_probability,confidence
5770,1,0,0.079867,0.420133
3879,1,0,0.082196,0.417804
27177,1,0,0.149546,0.350454
22991,1,0,0.152184,0.347816
5608,1,0,0.163987,0.336013
12864,1,0,0.165946,0.334054
12076,1,0,0.169816,0.330184
25838,1,0,0.171345,0.328655
13659,1,0,0.174066,0.325934
23810,1,0,0.174828,0.325172


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*


I reviewed all 52 features used by the final Random Forest model. The feature-name audit did not identify a feature that directly contains the target label, but several historical performance features require a measurement-window check.

The main potential risk is temporal leakage. Features such as impressions, clicks, sessions, CTR, average position, engagement, traffic, and their derived tiers contain historical performance information. These features are appropriate only if their measurement window is available before the period used to construct the `is_declining_label` outcome.

I therefore do not label these features as leakage solely because they are predictive. Instead, I classify them as requiring a window check. Page/content attributes such as word count and content age do not show an obvious leakage risk from their feature definitions.

The client-aware validation result also provides a useful caution: Precision@20 changed from 0.95 under a row-level random split to 0.65 under the client-aware split. This difference does not prove feature leakage, but it shows why validation design and feature timing both need to be considered before making a strong generalization claim.

Overall, the audit found no obvious direct target feature in the final feature names, while historical performance variables remain the main area requiring confirmation of their measurement windows.

In [27]:
# Section 3 — Leakage audit of the final feature set

# Show the exact features used by the model
feature_audit = pd.DataFrame({
    "Feature": feature_columns
})

# Categorize features for the audit
def leakage_category(feature):
    f = feature.lower()

    if any(term in f for term in [
        "target",
        "label",
        "declining",
        "trend"
    ]):
        return "Exclude / target-related"

    if any(term in f for term in [
        "impression",
        "click",
        "ctr",
        "position",
        "pageview",
        "session",
        "user",
        "engagement",
        "scroll",
        "traffic"
    ]):
        return "Historical performance — window check"

    if any(term in f for term in [
        "search_volume",
        "competition",
        "cpc"
    ]):
        return "Search-demand feature — timing check"

    return "Page/content attribute"

feature_audit["Audit category"] = (
    feature_audit["Feature"].apply(leakage_category)
)

feature_audit["Leakage status"] = np.where(
    feature_audit["Audit category"].eq(
        "Historical performance — window check"
    ),
    "Review measurement window",
    np.where(
        feature_audit["Audit category"].eq(
            "Search-demand feature — timing check"
        ),
        "Check availability at prediction time",
        "No obvious leakage from feature name"
    )
)

print("Total model features:", len(feature_audit))

display(feature_audit)

Total model features: 52


,Feature,Audit category,Leakage status
0,search_volume,Search-demand feature — timing check,Check availability at prediction time
1,competition,Search-demand feature — timing check,Check availability at prediction time
2,cpc,Search-demand feature — timing check,Check availability at prediction time
3,word_count,Page/content attribute,No obvious leakage from feature name
4,char_count,Page/content attribute,No obvious leakage from feature name
5,log_impressions_90d,Historical performance — window check,Review measurement window
6,log_clicks_90d,Historical performance — window check,Review measurement window
7,log_sessions_90d,Historical performance — window check,Review measurement window
8,log_ai_sessions_90d,Historical performance — window check,Review measurement window
9,days_with_impressions,Historical performance — window check,Review measurement window


In [28]:
# Section 3 — Leakage audit summary

audit_summary = (
    feature_audit["Leakage status"]
    .value_counts()
    .rename_axis("Leakage status")
    .reset_index(name="Feature count")
)

print("Leakage audit summary:")
display(audit_summary)

print("\nFeatures requiring historical-window review:")
historical_review = feature_audit[
    feature_audit["Leakage status"] == "Review measurement window"
]

display(historical_review[["Feature", "Audit category"]])

print("\nDirect target-related features detected:",
      (feature_audit["Audit category"] == "Exclude / target-related").sum())

Leakage audit summary:


,Leakage status,Feature count
0,No obvious leakage from feature name,25
1,Review measurement window,20
2,Check availability at prediction time,7



Features requiring historical-window review:


,Feature,Audit category
5,log_impressions_90d,Historical performance — window check
6,log_clicks_90d,Historical performance — window check
7,log_sessions_90d,Historical performance — window check
8,log_ai_sessions_90d,Historical performance — window check
9,days_with_impressions,Historical performance — window check
10,days_with_sessions,Historical performance — window check
13,ctr,Historical performance — window check
14,avg_position,Historical performance — window check
15,engagement_rate,Historical performance — window check
16,scroll_rate,Historical performance — window check



Direct target-related features detected: 0


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


### Original claim

The Random Forest model accurately predicts which pages should be prioritized for refresh and substantially outperforms the baseline.

### Safer claim

In this experiment, the Random Forest measured 0.65 Precision@20 on the client-aware holdout. The Week-4 baseline had a measured Precision@20 of 0.15 in the Week-4 evaluation. These results show an observed and directional difference in measured ranking performance, although the evaluations should not be treated as a perfectly matched head-to-head comparison unless the baseline is also evaluated on the same client-aware test set.

The Random Forest result can provide decision-support for prioritizing pages for review, but it should not be interpreted as a guarantee of future refresh outcomes or performance on unseen clients.

The client-aware result is also more conservative than the row-level random split, where Precision@20 measured 0.95. This reinforces that validation design affects measured performance and that stronger generalization claims would require further testing.

In [29]:
# Section 4 — Claim rewrite check

original_claim = (
    "The Random Forest model accurately predicts which pages "
    "should be prioritized for refresh and substantially "
    "outperforms the baseline."
)

safer_claim = (
    "In this experiment, the Random Forest measured 0.65 "
    "Precision@20 on the client-aware holdout. The Week-4 "
    "baseline had a measured Precision@20 of 0.15 in the "
    "Week-4 evaluation. These results show an observed and "
    "directional difference in measured ranking performance. "
    "The result can provide decision-support for prioritization, "
    "but it should not be interpreted as a guarantee of future "
    "refresh outcomes or performance on unseen clients."
)

safe_language = [
    "observed",
    "measured",
    "directional",
    "decision-support"
]

print("Original claim:")
print(original_claim)

print("\nSafer claim:")
print(safer_claim)

print("\nSafe-language check:")

for term in safe_language:
    print(f"{term}: {term in safer_claim.lower()}")

assert all(term in safer_claim.lower() for term in safe_language)
assert after_metrics["precision_at_20"] == 0.65

print("\nSection 4 check passed.")

Original claim:
The Random Forest model accurately predicts which pages should be prioritized for refresh and substantially outperforms the baseline.

Safer claim:
In this experiment, the Random Forest measured 0.65 Precision@20 on the client-aware holdout. The Week-4 baseline had a measured Precision@20 of 0.15 in the Week-4 evaluation. These results show an observed and directional difference in measured ranking performance. The result can provide decision-support for prioritization, but it should not be interpreted as a guarantee of future refresh outcomes or performance on unseen clients.

Safe-language check:
observed: True
measured: True
directional: True
decision-support: True

Section 4 check passed.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit the repo URL on the card